# Example of using the GPU on I-GUIDE Platform JupyterHub: LLM hosting and AI Agent

In this notebook we demonstrate how to host a tiny LLM in the I-GUIDE JypyterHub and make use of it for AI Agent prototyping.

Before we get started, let's verify the GPU information:


In [1]:
!nvidia-smi

Tue Mar 31 16:12:03 2026       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 525.85.05    Driver Version: 525.85.05    CUDA Version: 12.0     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  GRID A100X-10C      On   | 00000000:04:00.0 Off |                    0 |
| N/A   N/A    P0    N/A /  N/A |      0MiB / 10240MiB |      0%      Default |
|                               |                      |             Disabled |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

## Model deployment
For this demonstration, we will be using the Qwen3-1.7B model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "Qwen/Qwen3-1.7B"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16, 
    device_map="auto"
)



/cvmfs/iguide.purdue.edu/software/conda/geoai-gpu/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 311/311 [00:05<00:00, 53.63it/s]
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


## Talking to the LLM
Let’s try talking to the LLM we just deployed. You can interact with it just like you would with other LLM platforms you use every day.

In [3]:
from transformers import TextIteratorStreamer
import threading

prompt = "What is NSF I-GUIDE?"

messages = [
    {"role": "system", "content": "You are a helpful assistant. Answer concisely."},
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# Create streamer
streamer = TextIteratorStreamer(
    tokenizer,
    skip_prompt=True,
    skip_special_tokens=True
)

# Run generation in a separate thread
generation_kwargs = dict(
    **model_inputs,
    streamer=streamer,
    max_new_tokens=512,
    do_sample=True,
    temperature=0.3,
    top_p=0.9,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
)

thread = threading.Thread(target=model.generate, kwargs=generation_kwargs)
thread.start()

# Stream output token-by-token
for text_chunk in streamer:
    print(text_chunk, end="", flush=True)

<think>
Okay, the user is asking about NSF I-GUIDE. Let me start by recalling what I know. NSF stands for National Science Foundation, which is a U.S. federal agency that supports research and education. I-GUIDE is probably a program or initiative they're referring to.

I remember that the National Science Foundation has various programs, like the I-GUIDE (Innovative Graduate Education and Research in the Sciences) program. This program is designed to support graduate students in the sciences, providing them with funding and resources to conduct research. The goal is to enhance the quality of graduate education and research in the sciences.

Wait, I should check if there's a specific I-GUIDE program. Maybe it's part of the NSF's broader initiatives. The I-GUIDE program aims to improve the training of graduate students by offering financial support, mentorship, and resources. It's meant to help students develop skills and pursue research that has a significant impact on society.

I shou

## Creating an AI Agent with search capability
We observe that the tiny LLM failed to answer the question correctly. Let's create an AI Agent that combines the reasoning and generation capability of the LLM with search to see if we can have a chatbot that answers the question more accurately.

In [4]:
%pip install langchain-community langchain-huggingface duckduckgo-search -q
%pip install langchain langchain-openai -q

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [8]:
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_core.messages import HumanMessage
from transformers import pipeline

# 1. HF pipeline
hf_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1024,
    do_sample=True,
    temperature=0.7,
    pad_token_id=tokenizer.eos_token_id,
    return_full_text=False,
    max_length=None
)

llm = HuggingFacePipeline(pipeline=hf_pipe)
chat_model = ChatHuggingFace(llm=llm, tokenizer=tokenizer)

# 2. Tool
search_tool = DuckDuckGoSearchResults()

# 3. User query
query = "What is NSF I-GUIDE"

print("\n=== USER ===")
print(query)

# 4. Step 1: FORCE tool decision via prompt
decision_prompt = f"""
You are an assistant.

Decide whether the following question requires web search.

If YES, output ONLY:
SEARCH: <search query>

If NO, answer directly.

Question: {query}
"""

decision = chat_model.invoke([HumanMessage(content=decision_prompt)]).content

print("\n=== DECISION ===")
print(decision)

# 5. Step 2: Execute tool if needed
if "SEARCH:" in decision:
    search_query = decision.split("SEARCH:")[-1].strip()

    print("\n=== TOOL CALL ===")
    print("DuckDuckGoSearch:", search_query)

    tool_result = search_tool.run(search_query)

    print("\n=== TOOL RESULT ===")
    print(tool_result)

    # 6. Step 3: Final answer using tool result
    final_prompt = f"""
    Use the following information to answer the question.

    Question: {query}

    Information:
    {tool_result}

    Answer clearly:
    """

    final_answer = chat_model.invoke([HumanMessage(content=final_prompt)]).content

else:
    final_answer = decision

print("\n=== FINAL ANSWER ===")
print(final_answer)


=== USER ===
What is NSF I-GUIDE

=== DECISION ===
<think>
Okay, the user is asking about "NSF I-GUIDE." First, I need to figure out what this refers to. NSF is the National Science Foundation, which is a U.S. federal agency that supports research and education. The acronym I-GUIDE might stand for something specific.

I recall that the National Science Foundation has various programs and initiatives. One of their programs is the I-GUIDE, which I think stands for "Innovative Global Education and Discovery" or something similar. Wait, maybe it's related to the NSF's broader initiatives in education and research. Alternatively, I-GUIDE could be a specific program under the NSF.

I should check if there's a known program by that name. From what I remember, the NSF has programs like the NSF Graduate Research Fellowship Program (GRFP), but I-GUIDE isn't one of them. Maybe it's a newer program or a specific initiative. Another possibility is that I-GUIDE is part of the NSF's broader mission 

In [ ]:
decision_prompt = f"""
You are an assistant that decides whether to use a web search tool.

You must follow these rules:

1. If the question requires external or up-to-date information, output exactly one line in the following format:
SEARCH: <a short, effective search query derived from the question>

- Replace <a short, effective search query derived from the question> with a real query.
- Do NOT output placeholders like "<search query>".
- Do NOT include explanations or extra text.
- The search query should be concise (5–10 words) and directly relevant.

2. If the question can be answered without web search, answer directly in a concise way.

Output ONLY one of the two options.

Question:
{query}
"""